# E17 MagNet Composite Graphs

Author: Arush Arora

## Introduction

The current codebase focuses on delivering text $\mathbf{X}$ and positional encodings $\Psi$ through separate channels to the LLM, which seems to be confounding their interleaving process. $\Psi$, the GREPs, are defined as follows:

$$\Phi = \Phi\big(\mathbf{q};\, \mathbf{S}, \mathcal{H}\big) \qquad \mathbf{P} \coloneqq \mathbb{E}_{\mathbf{q} \sim \mathcal{N}(0,\, \mathbf{I}_D)}\big[\Phi\big] \qquad \mathbf{C} \coloneqq \mathbb{E}_{\mathbf{q}}\big[\Phi\Phi^\top\big] - \mathbf{P}\mathbf{P}^\top$$

$$\mathbf{\Psi} = \Phi\big(\mathbf{X} + \mathbf{P};\, \mathcal{T}\big) \quad \text{or} \quad \mathbf{\Psi} = \Phi\bigg((\mathbf{I}_{n + c} + \mathbf{C})\begin{bmatrix}\mathbf{X} \\ \mathbf{P}\end{bmatrix};\, \mathcal{T}\bigg)$$

In this experiment, we wish to work with the Composite Graph paradigm to determine whether the MagNet architecture from the paper [“MagNet: A Neural Network for Directed Graphs” (Zhang et al., 2021)](https://arxiv.org/pdf/2102.11391) can encode the directional spectral information essential to the full Composite Graphs architecture to correct the issues that were appearing during the latest iteration of the experiment in June.

Specifically, we propose the following instantiation of $\mathbf{S}$ per the paper, the normalized complex Hermitian adjacency matrix $\bar{\mathbf{H}}^{(r)}$, which can also be seen as the Normalized Magnetic Adjacency:

$$\mathbf{S} = \bar{\mathbf{H}}^{(r)} \coloneqq \mathbf{D}^{-1/2} \mathbf{A} \mathbf{D}^{-1/2} \odot \exp\big(i\mathbf{\Theta}^{(r)}\big)$$

$$\mathbf{\Theta}^{(r)} \coloneqq 2 \pi r (\mathbf{A} - \mathbf{A}^\top), \quad r \ge 0$$

We thus define the Composite Graphs architecture as such:

$$\big|\mathcal{V}_\text{Tx}\big| = c, \qquad \big|\mathcal{V}_\text{Sc}\big| = n$$

$$\mathcal{G} = \big(\mathcal{V}_\text{Tx} \cup \mathcal{V}_\text{Sc}, \mathcal{E}_\text{Tx} \cup \mathcal{E}_\text{Tx} \cup \mathcal{V}_\text{Cross} \cup \mathcal{V}_\text{Mention})$$

$$\mathcal{E}_\text{Cross} = \big\{\{u, v\} : \text{$u$ is in the node label for $v$},\ u \in \mathcal{V}_\text{Tx},\ v \in \mathcal{V}_\text{Sc}\big\}$$

$$\mathcal{E}_\text{Mention} = \big\{\{u, v\} : \text{$u$ and $v$ are in same node labels},\ u, v \in \mathcal{V}_\text{Tx}\big\}$$

## Mathematical Overview

### The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

#### Graph Convolutional Network (GNN)

The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Random Graph Positional Encodings (R-PEARL)

The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

### Sparse Graph Transformer

The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q \sim \mathcal{N}(0,\, \mathbf{I})}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

### Transformer

The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

### Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

## Setup

In [1]:
# %env CUDA_VISIBLE_DEVICES=0
%load_ext autoreload
%autoreload 2

In [ ]:
# Import modules.
import gc
import glob
import math
import copy
import wandb
import torch
import random
import pickle
import sympy as sp
import numpy as np
import networkx as nx

from typing import Union, Optional

import torch
from torch import Tensor, nn
import matplotlib.pyplot as plt
from IPython.display import display
from torch_geometric.data import Data
from torch_geometric.nn import ChebConv
from torch.nn.utils import clip_grad_norm_
from torch_geometric.typing import OptTensor
from torch_geometric.utils import to_networkx
from torch_geometric.loader import DataLoader
from torch.distributions import Cauchy, Normal
from torch_geometric.nn.dense.linear import Linear
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch_geometric.utils import to_dense_adj, to_networkx
from torch_geometric.utils.num_nodes import maybe_num_nodes
from torch_geometric.utils import coalesce, remove_self_loops

from prism.models.gt import GraphTransformer, SemanticGraphTransformer
from prism.models.gnn_llm import (_wire_rotate, build_injection_map,
                                  find_last_graph_scope, node_token_variants)
from prism.data import compact_prompt, data, utils

In [3]:
# Weights & Biases setup. Mirrors prism.training.train_v3._setup_wandb (project /
# name / tags / group + full-config logging), adapted for this notebook's hand-written
# train loops. Each training stage gets its own run, grouped/tagged by GNN type so the
# R-PEARL and GT variants of the same stage line up on one W&B dashboard. The helpers
# introspect the live optimizer / scheduler / loss objects so EVERY hyperparameter is
# logged without hand-maintaining a list.
WANDB_PROJECT = 'e17-composite-graphs'


def optimizer_hparams(optimizer):
    """Every optimizer setting: class name, shared defaults, and per-param-group values
    (LRs, betas, eps, weight_decay, ...) with the parameter tensors stripped out."""
    return {
        'optimizer': type(optimizer).__name__,
        'optimizer_defaults': dict(optimizer.defaults),
        'param_groups': [
            {k: v for k, v in g.items() if k != 'params'}
            for g in optimizer.param_groups
        ],
    }


def scheduler_hparams(scheduler):
    """Every LR-scheduler setting (or {'scheduler': None} when unused)."""
    if scheduler is None:
        return {'scheduler': None}
    keys = ('mode', 'factor', 'patience', 'threshold', 'threshold_mode',
            'cooldown', 'min_lrs', 'eps')
    return {
        'scheduler': type(scheduler).__name__,
        **{k: getattr(scheduler, k) for k in keys if hasattr(scheduler, k)},
    }


def loss_hparams(loss_fn):
    """Loss class, reduction, and pos_weight (resolved to plain Python)."""
    out = {'loss_fn': type(loss_fn).__name__,
           'reduction': getattr(loss_fn, 'reduction', None)}
    pos_weight = getattr(loss_fn, 'pos_weight', None)
    if pos_weight is not None:
        out['pos_weight'] = (pos_weight.detach().cpu().tolist()
                             if torch.is_tensor(pos_weight) else pos_weight)
    return out


def init_wandb(stage, hparams):
    """Start a W&B run for a training `stage`.

    Logs the FULL run config: the GNN construction kwargs (`model_hparams`, set in the
    GNN-instantiation cell) plus every optimizer / scheduler / loss / batching
    hyperparameter the caller assembles in `hparams`. `model_type` selects R-PEARL vs
    GT and drives the run name / tag / group. Returns the run; `reinit=True` so
    successive stages in one notebook session each open a fresh run.
    """
    return wandb.init(
        project=WANDB_PROJECT,
        name=f'{stage}_{model_type}',
        tags=[stage, model_type],
        group=model_type,
        config={'model_type': model_type, 'stage': stage,
                'model': model_hparams, **hparams},
        reinit='return_previous',
    )

In [4]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 3, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [ ]:
# Standard options.
ex_path   = '../data/n_30/gen/nav100_n30_gemma_data/split/test_graphs'
plan_path = '../data/n_30/gen/nav100_n30_gemma_data/generated_plans'
eval_path = '../data/n_100/gen/nav_n100_gemma_data/test_graphs'
save_path = '../data/pickle/e6_eval_graphs.pkl'
llm_path  = 'google/gemma-4-31B-it'
device    = 'cuda'

In [6]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(ex_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [7]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_001.html


## Experiments

### §1 Pretraining a GNN to Classify Text-Scene Node Pairings

We first hope to optimize a GNN (R-PEARL or Graph Transformer) to classify crosslink edges of the form $\{u, v\}$ where $u \in \mathcal{V}_\text{Tx}$ and $v \in \mathcal{V}_\text{Sc}$. Such a model will serve as a backbone pretrained model for fine-tuning on identifying node families given a node (including all other bucket text nodes, mention text nodes, and scene nodes). This architecture will instantiate the already implemented `GCN`, `RandomGNNPositionalEncodings`, and `GraphTransformer` architectures with the Magnetic Adjacency $\bar{\mathbf{H}}^{(r)}$ defined above.

#### Model Definitions

We first construct the MagNet architecture, mathematized below:

##### CReLU and Complex Linear Transformation

$$z \in \mathbb{C} \qquad \mathbf{X} \in \mathbb{C}^{N \times D}$$

$$\sigma(z) = \begin{cases}
    z, & \text{if } \arg(z) \in [- \pi / 2, \pi / 2] \\
    0, & \text{otherwise}
\end{cases}$$

$$\mathrm{CLin}_\phi\big(\mathbf{X}\big) = \mathrm{Lin}_\phi\Big(\mathrm{Re}\big(\mathbf{X}\big)\Big) + i\bigg[\mathrm{Lin}_\phi\Big(\mathrm{Im}\big(\mathbf{X}\big)\Big)\bigg]$$

In [8]:
def crelu(x: Tensor) -> Tensor:
    """Applies the complex rectified linear unit function."""
    return x * (x.real >= 0)


def clin(lin: Linear, x: Tensor) -> Tensor:
    """Applies a real-valued linear transformation to a complex tensor."""
    return torch.complex(lin(x.real), lin(x.imag))

##### Magnetic Laplacian and Chebyshev Convolution

$$A_s = \frac{1}{2}(\mathbf{A} + \mathbf{A}^\top)$$

$$\mathbf{\bar{H}}^{(r)} \coloneqq \mathbf{D}_s^{-1/2} \mathbf{A}_s \mathbf{D}_s^{-1/2} \odot \exp \left( i \mathbf{\Theta}^{(r)} \right)$$

$$\mathbf{\Theta}^{(r)} = 2 \pi r \, \mathrm{sgn} (\mathbf{A} - \mathbf{A}^\top), \quad r \ge 0$$

$$\bar{\mathbf{L}}^{(r)} \coloneqq \mathbf{I}_N - \bar{\mathbf{H}}^{(r)}$$

---

$$\mathbf{X}^{(0)} \in \mathbb{R}^{N \times D} \qquad \mathbf{S} = \mathbf{\bar{L}}^{(r)}$$

$$\mathbf{Y}^{(l)} = \sum_{k=1}^{K} \mathbf{Z}_k(\mathbf{X}^{(l)};\, \mathbf{S}) \mathbf{H}_k$$

$$\mathbf{S} \in \mathbb{C}^{N \times N} \quad \mathrm{spec}(\mathbf{S}) \subseteq [-1, 1]$$

$$\begin{align*}
\mathbf{Z}_1 &= \mathbf{X} \\
\mathbf{Z}_2 &= \mathbf{S} \mathbf{X} \\
\mathbf{Z}_k &= 2 \cdot \mathbf{S} \mathbf{Z}_{k-1} - \mathbf{Z}_{k-2}
\end{align*}$$

$$\mathbf{H}_k \in \mathbb{R}^{F \times G}$$

In [9]:
class MagChebConv(ChebConv):
    """The magnetic Chebyshev spectral graph convolutional operator."""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        K: int,
        normalization: Optional[str] = 'sym',
        bias: bool = True,
        r: float = 0.25,
        learn_r: bool = False,
        phase: str = 'binary',
        **kwargs,
    ):
        super().__init__(in_channels, out_channels, K, normalization, bias,
                         **kwargs)

        assert phase in ['binary', 'weight'], 'Invalid phase'

        self.phase, self.r_const = phase, r
        self.r_logit = nn.Parameter(
            torch.tensor(min(max(r, 1e-3), 0.249) / 0.25).logit(),
        ) if learn_r else None

    @property
    def r(self) -> Tensor:
        r"""The charge parameter :math:`r`, constrained to :math:`[0, 0.25]`.
        A hard clamp would leave no gradient at the upper end.
        """
        return self.r_const if self.r_logit is None else 0.25 * self.r_logit.sigmoid()

    def __norm__(
        self,
        edge_index: Tensor,
        num_nodes: Optional[int],
        edge_weight: OptTensor,
        normalization: Optional[str],
        lambda_max: OptTensor = None,
        dtype: Optional[int] = None,
        batch: OptTensor = None,
    ):
        num_nodes = maybe_num_nodes(edge_index, num_nodes)
        edge_index, edge_weight = remove_self_loops(edge_index, edge_weight)
        if edge_weight is None:
            edge_weight = torch.ones(edge_index.size(1), dtype=dtype,
                                     device=edge_index.device)

        row, col = edge_index[0], edge_index[1]
        sgn = edge_weight if self.phase == 'weight' else torch.ones_like(edge_weight)
        edge_index = torch.stack([torch.cat([row, col]), torch.cat([col, row])])
        edge_attr = torch.stack([edge_weight.repeat(2), torch.cat([sgn, -sgn])], 1)
        edge_index, edge_attr = coalesce(edge_index, edge_attr, num_nodes)

        edge_index, edge_weight = gcn_norm(edge_index, edge_attr[:, 0] / 2,
                                           num_nodes, add_self_loops=False)
        asym = edge_attr[:, 1].sign() if self.phase == 'binary' else edge_attr[:, 1]
        theta = (2 * math.pi) * self.r * asym

        return edge_index, torch.complex(edge_weight * theta.cos(),
                                         edge_weight * theta.sin())

    def forward(
        self,
        x: Tensor,
        edge_index: Tensor,
        edge_weight: OptTensor = None,
        batch: OptTensor = None,
        lambda_max: OptTensor = None,
    ) -> Tensor:

        edge_index, norm = self.__norm__(
            edge_index,
            x.size(self.node_dim),
            edge_weight,
            self.normalization,
            lambda_max,
            dtype=x.real.dtype,
            batch=batch,
        )

        Tx_0 = x
        Tx_1 = x  # Dummy.
        out = clin(self.lins[0], Tx_0)

        # propagate_type: (x: Tensor, norm: Tensor)
        if len(self.lins) > 1:
            Tx_1 = self.propagate(edge_index, x=x, norm=norm)
            out = out + clin(self.lins[1], Tx_1)

        for lin in self.lins[2:]:
            Tx_2 = self.propagate(edge_index, x=Tx_1, norm=norm)
            Tx_2 = 2. * Tx_2 - Tx_0
            out = out + clin(lin, Tx_2)
            Tx_0, Tx_1 = Tx_1, Tx_2

        if self.bias is not None:
            out = out + self.bias

        return out

##### MagNet

$$\Phi\Big(\mathbf{X};\, \mathbf{\bar{L}}^{(r)}, \mathcal{H}\Big) = \mathbf{X}'$$

$$\mathbf{X}^{\prime} = \mathbf{W}^\top \left[ \mathrm{Re}\Big(\mathbf{X}^{(L)}\Big) \, \Vert \,
        \mathrm{Im}\Big(\mathbf{X}^{(L)}\Big) \right]$$

$$\mathbf{X}^{(\ell)} = \sigma \Big( \mathbf{Y}^{(\ell - 1)} \Big)$$

In [10]:
class MagNet(nn.Module):
    """The magnetic graph neural network."""
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        num_layers: int,
        skip_connection: bool = False,
        dropout: float = 0.5,
        k: int = 3,
        r: float = 0.25,
        learn_r: bool = False,
        phase: str = 'binary',
    ):
        super().__init__()

        assert num_layers >= 2, 'MagNet requires at least 2 layers'

        dims = [in_channels] + [hidden_channels] * (num_layers - 1)
        self.convs = nn.ModuleList([
            MagChebConv(i, hidden_channels, k, r=r, learn_r=learn_r,
                        phase=phase) for i in dims
        ])
        self.norms = nn.ModuleList(
            [nn.LayerNorm(hidden_channels) for _ in range(num_layers - 2)])
        self.unwind = nn.Linear(2 * hidden_channels, hidden_channels)
        self.dropout = nn.Dropout(dropout)
        self.skip_connection = skip_connection
        self.embedding_dim = hidden_channels

    @property
    def r(self) -> Tensor:
        r"""The charge parameter :math:`r` of each layer."""
        return torch.stack([torch.as_tensor(conv.r) for conv in self.convs])

    def forward(self, data: Data) -> Tensor:
        device = next(self.parameters()).device
        x, edge_index = data.x.to(device), data.edge_index.to(device)
        edge_weight = getattr(data, 'edge_weight', None)
        if edge_weight is not None:
            edge_weight = edge_weight.to(device)
        batch = getattr(data, 'batch', None)
        if batch is not None:
            batch = batch.to(device)

        x = x_prev = x + 0j
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x_prev, edge_index, edge_weight, batch)
            if i < len(self.norms):
                x = torch.complex(self.norms[i](x.real), self.norms[i](x.imag))
            x = crelu(x) * self.dropout(torch.ones_like(x.real))
            if self.skip_connection and i > 0:
                x = x + x_prev
            x_prev = x

        x = self.convs[-1](x, edge_index, edge_weight, batch)
        return self.unwind(torch.cat([x.real, x.imag], dim=-1))

#### Numerical Visualizations with SymPy

We next wish to test out `MagNet` and verify that it produces coherent outputs. We will instantiate a raw MagNet and run it on all graphs.

In [11]:
# Prepare a graph from the data to be used in the GNN.
load_ex_graph = False

if load_ex_graph:
    with open(save_path, 'rb') as file:
        ex_graph = pickle.load(file)[4]
        N = ex_graph.num_nodes
else:
    ex_graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2], 'binary')

    N, D = ex_graph.num_nodes, 1024
    ex_graph.edge_index = ex_graph.edge_index.to(device)

    EPS = 1e-12
    MAX_LENGTH = 128
    g = to_networkx(ex_graph, to_undirected=True)
    ex_graph.nxg = g
    all_pairs = dict(nx.all_pairs_dijkstra(g, weight=None))
    delta_max = max(len(path) for target in all_pairs.values() for path in target[1].values())
    paths = torch.full((N, N, delta_max if delta_max < MAX_LENGTH else MAX_LENGTH), -1).long()
    dist = torch.full((N, N), float('inf'))
    for u, (lengths_u, paths_u) in all_pairs.items():
        for v, p in paths_u.items():
            dist[u, v] = lengths_u[v]
            p = (
                torch.tensor(p, device=device).long() if len(p) < MAX_LENGTH 
                else torch.full((MAX_LENGTH,), -1, device=device).long()
            )
            paths[u, v, 0:len(p)] = p
            paths[v, u, 0:len(p)] = p.flip(0)
    dist.fill_diagonal_(EPS)
    ex_graph.diameter = delta_max
    ex_graph.paths = paths.to(device)
    ex_graph.dist = dist.to(device)

    # Topology only: BFS hop counts, never weighted, so metric distances in `dist`
    # can never reach the blurry-vision mask.
    hops = torch.full((N, N), float('inf'))
    for u, lengths_u in nx.all_pairs_shortest_path_length(g):
        for v, h in lengths_u.items():
            hops[u, v] = h
    assert torch.equal(hops[hops.isfinite()], hops[hops.isfinite()].round()), \
        'graph.hops must stay integral (unweighted BFS)'
    ex_graph.hops = hops.to(device)

    # Dense adjacency (bool).
    ex_graph.adj = to_dense_adj(
        ex_graph.edge_index, max_num_nodes=N
    ).squeeze(0).bool().to(device)

# Show the shortest paths matrix of a node in the graph.
node1 = random.randint(0, N - 1)
node2 = random.randint(0, N - 1)
render_matrix(ex_graph.paths[node1, node2][None, :], sig_figs=0)

Matrix([[71, 72, 29, 103, 48, -1, -1]])

In [ ]:
# Instantiate a GNN. `model_type` / `model_hparams` are exposed at module scope so
# init_wandb can log the GNN config; create_gnn writes model_hparams as it builds.
def create_gnn(model_type: str, **overrides):
    global model_hparams
    model_hparams = dict(
        num_layers=3,
        pe_hidden_channels=256,
        pe_num_layers=5,
        d_model=1024,
        heads=8,
        num_samples=320,
        dropout=0.1,
        k_pe=3,
        k_gt=2,
        eps=1e-6,
        use_layer_norm=True,
        # E_q is taken INSIDE R-PEARL, so the probe stack Φ — and with it the second
        # moment C = E_q[ΦΦᵀ] - ΨΨᵀ — is read off before the blocks. See §3.
        pe_pool='pe',
        directed=True
    )
    model_hparams.update(overrides)
    gnn = GraphTransformer(**model_hparams)
    gnn.out_features = gnn.d_model
    return gnn


model_type = 'gt'
gnn = create_gnn(model_type)
gnn

In [13]:
# Define a function to assemble the composite graph of a scene graph and its prompt.
def build_composite_graph(
    source,
    tokenizer,
    tasks=None,
    plan_files=(),
    edge_weights: str = 'binary',
    include_edges: bool = False,
    include_tools: bool = False,
    cycle_weight: float = 1.0,
    cycle_directed: bool = True,
    cycle_causal: bool = False,
    crosslink_weight: float = 1.0,
    crosslink_mention_to_node: bool = True,
    context_window: int = 1024,
    crosslink_bidirectional: bool = False,
    anchor: bool = False,
    anchor_weight: float = 1.0,
    device=device,
) -> Data:
    """
    Assembles the composite graph of a scene graph and the text it is served to the
    LLM in.

    Args:
        source: an eval data_gen file, a scene graph dict, or an already-built PyG
            scene graph (`ex_graph`, which carries its own `raw_scene_graph`).
        tokenizer: the LLM tokenizer, whose tokenization of the prompt is V_Tx.
        tasks: the task strings the eval prompt states; read off `source` when it is
            a data_gen file, and required otherwise.
        plan_files: the generated plans of that same graph, as a glob or a list of
            paths. Given, their conversation is the text layer; omitted, the eval
            prompt is (generation turn open, no answers).
        edge_weights: 'binary' for a plain adjacency or 'gaussian' for the train-time
            affinity, as in `scene_graph_dict_to_pyg`.
        include_edges: state the edge bullets in the scene graph block of the text.
        include_tools: document the SPINE API and keep action-list plans in the text.
        cycle_weight: the weight of every E_Tx edge.
        cycle_directed: keep E_Tx one-directional, so that S_Tx stays circulant.
        cycle_causal: transpose E_Tx, so that a token reads its PREDECESSOR. Attention
            follows the edge direction, so this is what makes the prefill causal.
        crosslink_weight: the weight of every E_Cross edge.
        crosslink_mention_to_node: point each mention token at its scene node.
        context_window: the size of V_Tx, counted from the last scene-graph block on.
            A shorter prompt is taken whole; a longer one is truncated to its FIRST
            `context_window` tokens, so the graph block itself is never the part cut.
        crosslink_bidirectional: point each scene node back at its mention tokens too.
            That pair is then symmetric, so sgn(A - Aᵀ) — and with it the crosslink's
            phase — vanishes, and a token becomes readable from a LATER token through
            its scene node, which is exactly what §2's causal prefill forbids.
        anchor: bond one extra node t_0 → anchor → v_0, joining V_Tx and V_Sc whatever
            E_Cross covers; t_0 is the token that opens the scene-graph block. The anchor
            is the LAST row, and is neither a token node nor a scene node.
        anchor_weight: the weight of both anchor bonds.
        device: the device the assembled edges are built on.

    Returns:
        A PyG `Data` over c + n (+ 1) nodes, text first, whose c text nodes are the
        prompt FROM the last scene-graph block on, carrying `edge_index`,
        `edge_weight`, `is_token`, `num_token_nodes`, `num_scene_nodes`,
        `injection_map`, `node_names`, and `input_ids`. The text-first layout is the
        one `covariance_token_block` reads.
    """
    # Ported from `prism.models.composite_graph`, deleted in b2fc7bd with the legacy archs.
    if isinstance(source, Data):
        graph_dict, scene = source.raw_scene_graph, source
    else:
        payload = utils.try_load_json(source) if isinstance(source, str) else source
        graph_dict = payload.get('graph', payload)
        if tasks is None and 'tasks' in payload:
            tasks = [t['task'] for t in payload['tasks']]
        scene = utils.scene_graph_dict_to_pyg(graph_dict, edge_weights=edge_weights)

    # V_Tx is the tokenized plan conversation when plans are given, else the eval prompt.
    rollouts = [utils.try_load_json(p) for p in
                (sorted(glob.glob(plan_files)) if isinstance(plan_files, str) else plan_files)]
    if rollouts:
        messages = compact_prompt.assemble_training_conversation(
            rollouts, include_edges=include_edges, include_tools=include_tools)
    elif tasks is not None:
        messages = compact_prompt.format_eval_messages(
            graph_dict, tasks, include_edges=include_edges, include_tools=include_tools)
    else:
        raise ValueError('an eval composite graph needs `tasks` (or a data_gen file carrying them)')
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=not rollouts, return_dict=False)

    # V_Tx BEGINS at the last scene-graph block: everything before it is system prompt and
    # ICL graphs, which no crosslink reaches and which the GT is never asked about. What
    # follows is the context window the LLM is served, and no more.
    dropped = find_last_graph_scope(input_ids, tokenizer)
    input_ids = input_ids[dropped:dropped + context_window]

    # The crosslinks reuse the LLM's own injection map, now scoped by the truncation.
    injection_map = build_injection_map(
        input_ids, node_token_variants(scene.node_names, tokenizer))

    c, n_scene, n_anchor = len(input_ids), scene.num_nodes, int(anchor)
    rows, cols, vals = [], [], []

    def _add(src, dst, w):
        rows.append(src)
        cols.append(dst)
        vals.append(torch.full((src.numel(),), float(w), dtype=torch.float32, device=device))

    # E_Tx: the directed cycle, whose wraparound makes S_Tx circulant; MagNet carries its
    # direction as the phase Θ = 2πr sgn(A - Aᵀ), so transposing it only conjugates Ψ.
    i = torch.arange(c, device=device)
    nbr = (i - 1) % c if cycle_causal else (i + 1) % c
    _add(i, nbr, cycle_weight)
    if not cycle_directed:
        _add(nbr, i, cycle_weight)

    # E_Sc: the scene edges shifted by +c, weighted by W_Sc (all ones under 'binary').
    scene_ei = scene.edge_index.to(device)
    scene_ew = getattr(scene, 'edge_weight', None)
    if scene_ei.numel():
        rows.append(scene_ei[0] + c)
        cols.append(scene_ei[1] + c)
        vals.append((torch.ones(scene_ei.shape[1]) if scene_ew is None else scene_ew)
                    .to(device=device, dtype=torch.float32))

    # E_Cross: each mention token READS its scene node, and never the reverse, so that no
    # path carries token content back into the text layer. MagNet still sees both sides.
    if crosslink_mention_to_node:
        for node_idx, spans in injection_map.items():
            tok = torch.tensor(sorted({t for s, e in spans for t in range(s, min(e, c))}),
                               device=device, dtype=torch.long)
            if not tok.numel():
                continue
            node = torch.full_like(tok, c + node_idx)
            _add(tok, node, crosslink_weight)
            if crosslink_bidirectional:
                _add(node, tok, crosslink_weight)

    # The anchor bond: E_Tx is a cycle and E_Sc is connected, so this one extra node makes
    # the WHOLE of G connected — and with it every effective resistance R(u, v) finite.
    if anchor:
        assert n_scene, 'the anchor bond needs a scene node to bond to'
        a = torch.tensor([c + n_scene], device=device)
        _add(torch.zeros_like(a), a, anchor_weight)
        _add(a, torch.full_like(a, c), anchor_weight)

    composite = Data(
        x=torch.zeros(c + n_scene + n_anchor, 1, device=device),
        edge_index=torch.stack([torch.cat(rows), torch.cat(cols)]),
        edge_weight=torch.cat(vals),
        num_nodes=c + n_scene + n_anchor,
    )
    composite.is_token = torch.zeros(c + n_scene + n_anchor, dtype=torch.bool, device=device)
    composite.is_token[:c] = True
    composite.num_token_nodes, composite.num_scene_nodes = c, n_scene
    composite.injection_map = injection_map
    composite.node_names = scene.node_names
    composite.input_ids = input_ids

    # An uncrosslinked scene node is a component of its own, so coverage is measured.
    linked = [k for k, v in injection_map.items() if any(s < min(e, c) for s, e in v)]
    print(f"Composite Graph: {c} text nodes, {n_scene} scene nodes"
          f"{' + anchor' if anchor else ''}, {composite.edge_index.shape[1]} edges "
          f"| Crosslinked: {len(linked)}/{n_scene} | Window: {c}/{context_window} "
          f"| Preamble dropped: {dropped}")
    return composite

In [14]:
# Build the composite graphs of the loaded example and feed them to an untrained GNN.
tokenizer = AutoTokenizer.from_pretrained(llm_path)

# The eval arm reads the data_gen file alone; the training arm adds its generated plans.
eval_composite = build_composite_graph(
    graph_file_by_name[graph_file], tokenizer, device=device
)
train_composite = build_composite_graph(
    graph_file_by_name[graph_file], tokenizer, device=device,
    plan_files=f"{plan_path}/sample_{graph_file.split('_')[-1]}_*.json",
)

# `directed=True` swaps R-PEARL's TAGConv backbone for MagNet, so S = H̄^(r); untrained.
model_type = 'gt'
gnn = create_gnn(model_type).to(device).eval()

for name, composite in (('Eval', eval_composite), ('Train', train_composite)):
    with torch.no_grad():
        out = gnn(composite).to(device)

    # Ψ is shift-invariant on E_Tx alone, so text rows decorrelate only at the crosslinks.
    text_pe, scene_pe = out[composite.is_token], out[~composite.is_token]
    shift_cos = torch.cosine_similarity(text_pe[:-1], text_pe[1:], dim=1).mean()
    print(f"{name} Composite: \n Text PE: {text_pe.norm(dim=1).mean():>0.4f}, "
          f"Scene PE: {scene_pe.norm(dim=1).mean():>0.4f} | Shift Cosine: {shift_cos:>0.4f}")

    _, _, V = torch.pca_lowrank(out.cpu(), q=10, center=True)
    out = (out - out.mean(dim=0)) @ V.to(device)
    display(render_matrix(torch.cat([out[composite.is_token][:3],
                                     out[~composite.is_token][:3]])))

Composite Graph: 1449 text nodes, 104 scene nodes, 3337 edges | Crosslinked: 104/104
Composite Graph: 4388 text nodes, 104 scene nodes, 8806 edges | Crosslinked: 104/104
Eval Composite: 
 Text PE: 24.3708, Scene PE: 24.3709 | Shift Cosine: 0.9723


Matrix([
[-6.23, 2.06, -0.0562, -0.0794, -0.00553,    0.34,  -0.646, -0.377,   0.137,    0.15],
[-6.35, 2.15,    0.05,   -0.07,  -0.0102,   0.259,  -0.171, 0.0887,   0.181,    -0.2],
[-6.38, 2.07,  0.0508,  -0.118,   -0.209,   0.315,   0.374,  0.234,  -0.214, -0.0952],
[ 24.0, 4.68,   0.956,   -5.83,     6.91,  -0.161,  -0.197, -0.205, -0.0224,   0.204],
[ 23.6, 4.44,    1.14,   -5.78,     7.11, 0.00659, -0.0049, 0.0452,  0.0833,  -0.133],
[ 23.6, 4.52,     1.3,   -5.67,     7.25,  -0.029, -0.0895,  0.225,  -0.113, -0.0649]])

Train Composite: 
 Text PE: 24.3708, Scene PE: 24.3710 | Shift Cosine: 0.9503


Matrix([
[ 7.78, 0.763,   0.443, 0.583,   0.26,  0.0232,  0.0705,    0.11,  -0.0709,  -0.172],
[ 7.95,  1.15,  0.0492, 0.398,  0.491, -0.0803,  0.0222,  -0.132,   -0.217,   0.352],
[ 7.74, 0.976, -0.0296, 0.346,  0.531, -0.0329, -0.0145,   0.195, 0.000417,   0.142],
[-21.0,  16.8, -0.0747, -2.62, -0.838,   -5.61,  -0.287, -0.0412,  -0.0104,  0.0804],
[-18.4,  16.0,  -0.889, -5.73, -0.534,    -5.4,   0.736,   0.268,    0.093, -0.0293],
[-20.5,  16.6,  -0.272, -3.51, -0.786,   -5.41,   0.214,  0.0585,  -0.0113,  0.0978]])

### §2 Pretraining a GT to Reproduce RoPE-Rotated Word Embeddings

We now wish to optimize a Graph Transformer with a MagR-PEARL backbone to reproduce the RoPE-rotated word embeddings of the text nodes $\mathcal{V}_\text{Tx}$ of the composite graph, given the plans generated over the training dataset. Such a model will demonstrate that the circulant text layer $\mathcal{E}_\text{Tx}$ carries the LLM's own notion of sequence position, so that $\mathbf{X}$ and $\mathbf{\Psi}$ are expressed in one basis before the GT is asked to serve the LLM. The equations to represent this procedure are below:

$$\mathbf{Y}^{(\tau)} = \Phi\Big(\mathbf{X}^{(\tau)} + \mathbb{\hat{E}}_{\mathbf{q} \sim \mathcal{N}(0,\, \mathbf{I})}\big[\Phi\big(\mathbf{q};\, \bar{\mathbf{L}}^{(r)}, \mathcal{H}\big)\big];\, \mathcal{T}\Big) \qquad r \in [0,\, 1/4]$$

$$\mathbf{X}^{(\tau)} = \begin{bmatrix} \mathbf{\tilde{X}}_{1:\tau} \\ \mathbf{0}_{(c - \tau + n) \times d} \end{bmatrix} \qquad \mathbf{\tilde{X}} = \begin{bmatrix}
\mathbf{W}_e^\top \mathbf{e}_1 & \overset{\mathbf{W}_e^\top \mathbf{e}_t}{\cdots} & \mathbf{W}_e^\top \mathbf{e}_c
\end{bmatrix}^\top \in \mathbb{R}^{c \times d}$$

$$\mathbf{\hat{Y}}^{(\tau)} = \begin{bmatrix}
\mathbf{R}^{(d)}_{\Theta,\ 1}\mathbf{\tilde{x}}_1 & \overset{\mathbf{R}^{(d)}_{\Theta,\ t}\mathbf{\tilde{x}}_t}{\cdots} & \mathbf{R}^{(d)}_{\Theta,\ \tau}\mathbf{\tilde{x}}_\tau
\end{bmatrix}^\top \qquad \mathbf{R}^{(d)}_{\Theta,\ m} = \bigoplus_{i = 1}^{d / 2} \mathbf{R}(m\theta_i), \quad \theta_i = \frac{1}{\beta^{2(i - 1)/d}}$$

$$\text{Mean-Squared Error Loss: } \mathcal{L}\big(\mathbf{Y}^{(\tau)}, \mathbf{\hat{Y}}^{(\tau)}\big) = \frac{1}{\tau}\sum_{t = 1}^{\tau} \big\Vert \mathbf{y}^{(\tau)}_t - \mathbf{\hat{y}}_t \big\Vert_2^2$$

The training is autoregressive over the prefill: $\mathbf{X}^{(\tau)}$ carries the first $\tau$ word embeddings of the sequence and zeros everywhere else, and $\tau$ runs from $1$ to $c$, so that one token at a time is added to $\mathcal{V}_\text{Tx}$ exactly as it would arrive during generation. The sweep costs nothing, because the topology already enforces it. Attention follows the edge direction, so $\mathcal{E}_\text{Tx}$ is laid down as the TRANSPOSED cycle $t \to t - 1$ and $\mathcal{E}_\text{Cross}$ points from a mention token to its scene node and never back; no path of any depth then carries token content forward, row $t$ of $\mathbf{Y}^{(\tau)}$ is the same for every $\tau \ge t$, and the single forward at $\tau = c$ IS the whole sweep. The one exception is the wraparound $0 \to c - 1$, which the first $k$ rows read and which is exactly what keeps $\mathbf{S}_\text{Tx}$ circulant. Transposing the cycle costs the GREPs nothing either: $\bar{\mathbf{H}}^{(r)}$ symmetrizes $\mathbf{A}$ and keeps the direction only as the sign of $\Theta^{(r)}$, so $\mathbf{S} \mapsto \mathbf{S}^\top$ merely conjugates $\mathbf{\Psi}$.

Only the text rows are supervised, and the scene layer stays a live distractor: $\mathcal{V}_\text{Tx}$ still reads $\mathcal{V}_\text{Sc}$ through $\mathcal{E}_\text{Cross}$, and the scene rows still carry their own $\mathbf{\Psi}$, so the GT must learn to suppress a signal it can see — it simply cannot see the future through it.

The charge $r$ is no longer pinned at $1/4$ but learned, one per `MagChebConv` layer, through the sigmoid reparameterization $r = \tfrac{1}{4}\sigma(\rho)$ that keeps it inside $[0, 1/4]$ with a gradient at the upper end. Since $\Theta^{(r)} = 2 \pi r \, \mathrm{sgn}(\mathbf{A} - \mathbf{A}^\top)$ is the only place the direction of $\mathcal{E}_\text{Tx}$ enters, the learned $r$ is precisely the amount of directional phase the composite graph needs in order to express $\mathbf{R}^{(d)}_{\Theta,\ m}$.

In [ ]:
# Init variables. §2 alone needs the LLM's word embeddings, so the 31B load — and every
# constant read off it — is skipped when the RoPE stage is off; §3 reads topology only.
plans_per_graph = 10
train_prop, val_prop = 0.6, 0.2
train_rope = False

# Configure the datasets; the split is over SCENE graphs, so no plan leaks across.
keys = random.sample(list(samples_by_graph.keys()), k=len(samples_by_graph))
train_num, val_num = int(len(keys) * train_prop), int(len(keys) * val_prop)
train_keys = keys[:train_num]
val_keys = keys[train_num:train_num + val_num]
test_keys = keys[train_num + val_num:]

if train_rope:
    # The LLM's word-embedding table; its 60 decoder layers are dropped straight after.
    llm = AutoModelForCausalLM.from_pretrained(llm_path, dtype='auto', device_map='auto')
    text_config = llm.config.get_text_config()
    embed_table = llm.get_input_embeddings().weight.detach().to(device)
    del llm
    gc.collect()
    torch.cuda.empty_cache()

    # The local (sliding-attention) basis is the one that tiles d: the global one is
    # 512-wide and would leave half a rotation plane over the 5376-wide embedding.
    D, HEAD_DIM = embed_table.shape[1], text_config.head_dim
    BETA = text_config.rope_parameters['sliding_attention']['rope_theta']
    INV_FREQ = 1 / BETA ** (torch.arange(0, HEAD_DIM, 2, device=device).float() / HEAD_DIM)
    assert D % HEAD_DIM == 0, 'the attention head width must tile the embedding width'


def rope(x: Tensor) -> Tensor:
    """Rotates every head-width block of the rows of `x` by the LLM's own RoPE."""
    c, blocks = x.shape[0], D // HEAD_DIM
    freqs = torch.arange(c, device=device).float()[:, None] * INV_FREQ[None, :]
    # Gemma pairs channel n with n + d/2, so R^(d) is its rotate_half up to relabeling.
    emb = torch.cat((freqs, freqs), dim=-1)[None]
    out = _wire_rotate(x.view(1, c, blocks, HEAD_DIM).transpose(1, 2), emb.cos(), emb.sin())
    return out.transpose(1, 2).reshape(c, D)


# Preprocess the data.
def generate_data(keys, **kwargs):
    graphs = []
    for key in keys:
        plans = sorted(glob.glob(f"{plan_path}/sample_{key.split('_')[-1]}_*.json"))
        for plan in plans[:plans_per_graph]:
            graph = build_composite_graph(graph_file_by_name[key], tokenizer,
                                          plan_files=[plan], cycle_causal=True,
                                          device=device, **kwargs)
            graph.input_ids = torch.tensor(graph.input_ids, device=device)
            graphs.append(graph)

    return graphs


def seed_graph(graph, prefix: int = None):
    """Seeds the first `prefix` tokens of the prefill on V_Tx and their rotations as
    the target; the rest of V_Tx, and the whole of V_Sc, stay zero and unsupervised.
    The text nodes come first, so the filled rows are exactly `[:prefix]`."""
    prefix = graph.num_token_nodes if prefix is None else prefix
    x = torch.zeros(graph.num_nodes, D, device=device)
    x[:prefix] = embed_table[graph.input_ids[:prefix]].float()
    graph.x = x
    graph.y = rope(x[:prefix])
    graph.prefix = prefix

    return graph


train_graphs = generate_data(train_keys)
val_graphs = generate_data(val_keys)
test_graphs = generate_data(test_keys)

In [ ]:
# Train the Graph Transformer to reproduce the RoPE-rotated word embeddings.
batch_size = 4
val_freq = 5
epochs = 50
es_patience = 5

if train_rope:
    # Y = Φ(X + P; T) at the LLM's embedding width, so no readout stands before the target.
    model_type = 'gt'
    gnn = create_gnn(model_type, d_model=D, node_feature_dim=D,
                     fuse_node_features=True).to(device)
    charges = [p for n, p in gnn.named_parameters() if n.endswith('r_logit')]
    weights = [p for n, p in gnn.named_parameters() if not n.endswith('r_logit')]
    assert len(charges) == 1, 'MagNet shares one charge across its layers, and it is not learnable'

    # Gating check: the causal cycle and the one-directional crosslinks leave row t reading
    # only rows <= t, so ONE forward over the whole prefill IS the sweep over every τ. The
    # check compares the two against each other, holding Ψ fixed so only X varies.
    # The seam is the STACK's radius: num_layers blocks each reach k_gt hops, so rows
    # within num_layers * k_gt of row 0 read the wraparound edge 0 → c - 1 and are skipped.
    tau, seam = 64, model_hparams['num_layers'] * model_hparams['k_gt']
    gnn.cache_pe, _ = True, gnn.eval()
    with torch.no_grad():
        full = gnn(seed_graph(train_graphs[0]))[seam:tau]
        part = gnn(seed_graph(train_graphs[0], tau))[seam:tau]
    gnn.cache_pe = False
    gnn.invalidate_cache()
    assert torch.allclose(full, part, atol=1e-4), 'row t of the prefill still reads past τ'
    print(f"Prefill causal to row {tau} past the seam of {seam} | "
          f"Learnable charges: {gnn.pe_model.pe_gcn.r.detach().tolist()}")


def test_loop_rope(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss, cosine, error_norm = 0, 0, 0

    with torch.no_grad():
        for idx in order.tolist():
            graph = seed_graph(dataloader.dataset[idx])
            preds = model(graph)[:graph.prefix]
            test_loss += loss_fn(preds, graph.y).item()
            cosine += torch.cosine_similarity(preds, graph.y, dim=1).mean().item()
            error_norm += ((preds - graph.y).norm() / graph.y.norm()).item()

    test_loss /= size
    cosine /= size
    error_norm /= size
    charge = model.pe_model.pe_gcn.r
    print(f"Test Error: \n Cosine: {(100*cosine):>0.1f}%, Rel Err: {error_norm:.3f} "
          f"| Charge: {charge.mean().item():.4f} | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/cosine': cosine,
            f'{wandb_prefix}/error_norm': error_norm,
            f'{wandb_prefix}/charge': charge.mean().item(),
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, cosine


def train_loop_rope(train_dataloader, val_dataloader, test_dataloader, model,
              loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    best_val, best_state, bad_runs = float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('rope_regression', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        'plans_per_graph': plans_per_graph,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq + 1}\n=============")
            val_loss, _ = test_loop_rope(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if val_loss < best_val - 1e-3:
                best_val, bad_runs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model.train()

        print(f"=============\nEpoch #{i + 1}\n=============")
        optimizer.zero_grad()
        pending = 0
        order = torch.randperm(size)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss. One forward over the completed prefill covers
            # every τ at once, since row t of a causal prefill cannot read past itself.
            graph = seed_graph(train_dataloader.dataset[idx])
            preds = model(graph)[:graph.prefix]
            loss = loss_fn(preds, graph.y)

            # Backpropagation.
            (loss / batch_size).backward()
            pending += 1

            # Optimization and results.
            if pending == batch_size:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                pending = 0

                global_step += 1
                loss, current = loss.item(), j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'train/charge': model.pe_model.pe_gcn.r.mean().item(),
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

        if pending:
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            pending = 0

    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test the finished model.
    test_loop_rope(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


loss_fn = nn.MSELoss()
train_dataloader = DataLoader(train_graphs, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs, batch_size=batch_size)
if train_rope:
    optimizer = torch.optim.AdamW([
        {'params': weights, 'lr': 3e-4},
        # Decay on a charge logit pulls r to 1/8, not to 0, so the charges opt out.
        {'params': charges, 'lr': 1e-2, 'weight_decay': 0.0},
    ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_rope(train_dataloader, val_dataloader, test_dataloader, gnn,
                    loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler
    gc.collect()

### §3 Pretraining a GT to Reproduce the Composite Graph's Effective Resistances

We now wish to optimize the MagR-PEARL backbone so that the SECOND moment of its probe response is a metric embedding of the composite graph. Where §2 pinned the first moment $\mathbf{\Psi}$ to the LLM's own basis, this stage fixes the geometry of the fluctuation around it — and the fluctuation is what the composite architecture actually injects, since the readout of the introduction is $\Phi\big((\mathbf{I}_{n + c} + \mathbf{C})[\mathbf{X}; \mathbf{P}];\, \mathcal{T}\big)$ and its token block $\mathbf{C}_\text{tok}$ is the only place a token-to-token structural distance can enter. The equations to represent this procedure are below:

$$\Phi(\mathbf{q}) = \Phi\big(\mathbf{q};\, \bar{\mathbf{L}}^{(r)}, \mathcal{H}\big) \qquad \mathbf{\Psi} = \mathbb{\hat{E}}_{\mathbf{q} \sim \mathcal{N}(0,\, \mathbf{I})}\big[\Phi\big] \qquad \mathbf{C} = \mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi\Phi^\top\big] - \mathbf{\Psi}\mathbf{\Psi}^\top$$

$$R_\text{eff}(u,\, v) = \big(\mathbf{e}_u - \mathbf{e}_v\big)^H \big(\bar{\mathbf{L}}^{(r)}\big)^\dagger \big(\mathbf{e}_u - \mathbf{e}_v\big) = \bar{L}^\dagger_{uu} + \bar{L}^\dagger_{vv} - 2\operatorname{Re}\big(\bar{L}^\dagger_{uv}\big) \in \mathbb{R}_{\ge 0}$$

$$\text{Loss: } \mathcal{L}\big(\mathbf{C}\big) = \sum_{n = 1}^{N}\sum_{m = 1}^{N} \Big(C_{nn} + C_{mm} - 2C_{nm} - \alpha R_\text{eff}\big(u(n),\, v(m)\big)\Big)^2, \qquad N = c + n_\text{Sc} + 1$$

With `pe_pool='pe'` the probe expectation is taken INSIDE R-PEARL, so the stack the covariance is formed over is $\Phi$ itself and not its image $\Phi' = \Phi(\Phi(\mathbf{q}); \mathcal{T})$ under the blocks; $\mathbf{C}$ is then exactly the $\mathbb{E}[\Phi\Phi^H] - \mathbf{\Psi}\mathbf{\Psi}^H$ of the figure, and `covariance_token_block` already forms it. One consequence is worth stating plainly: the transformer blocks stand DOWNSTREAM of $\mathbf{C}$ and take no gradient from this loss, so what this stage trains is the MagNet backbone and R-PEARL's projection — the blocks are §2's business. The quantity being fitted is
$$C_{nn} + C_{mm} - 2C_{nm} = \mathbb{\hat{E}}_{\mathbf{q}}\Big[\big\Vert\Phi_n - \Phi_m\big\Vert_2^2\Big] - \big\Vert\boldsymbol{\psi}_n - \boldsymbol{\psi}_m\big\Vert_2^2 = \mathbb{\hat{E}}_\mathbf{q}\Big[\big\Vert\big(\Phi_n - \Phi_m\big) - \big(\boldsymbol{\psi}_n - \boldsymbol{\psi}_m\big)\big\Vert_2^2\Big],$$
the variance across probes of the difference of two rows: non-negative by construction, zero exactly when two nodes answer every probe alike, and blind to the mean $\mathbf{\Psi}$ that §2 already spent. Rows $n$ and $m$ are nodes of the composite graph under the text-first layout, so $u(n) = n$ and the token block $\mathbf{C}_\text{tok} = \mathbf{C}_{1:c,\, 1:c}$ of the figure is the leading sub-block; the loss supervises all of $\mathbf{C}$ and the cells below report $\mathbf{C}_\text{tok}$ separately.

$R_\text{eff}$ is real by the Hermitian pairing rather than by fiat. $\mathbf{\Theta}^{(r)}$ is skew-symmetric, so $\bar{\mathbf{H}}^{(r)}$ — and with it $\bar{\mathbf{L}}^{(r)}$ and its pseudoinverse — is Hermitian; the literal transcription $\bar{L}^\dagger_{uu} + \bar{L}^\dagger_{vv} - 2\bar{L}^\dagger_{uv}$ is then COMPLEX, carrying a stray $-2i\operatorname{Im}\big(\bar{L}^\dagger_{uv}\big)$, whereas the quadratic form pairs $\bar{L}^\dagger_{uv}$ with $\bar{L}^\dagger_{vu} = \overline{\bar{L}^\dagger_{uv}}$ and that imaginary part cancels exactly. Since $\operatorname{spec}\big(\bar{\mathbf{H}}^{(r)}\big) \subseteq [-1, 1]$ (Zhang et al., Theorem 2), $\bar{\mathbf{L}}^{(r)} \succeq 0$ and the form is non-negative with $R_\text{eff}(u,\, u) = 0$: both sides of the loss are squared distances.

$\alpha$ is FIXED and not learned. With $\alpha$ free the pair $(\mathbf{C},\, \alpha) \to (\mathbf{0},\, 0)$ drives the loss to zero, so the only scale that carries information is the one held constant. The loss has a floor that is set by $M$ rather than by the model: $\mathbf{C}$ is a sample covariance of $M$ probe responses, so $\operatorname{rank}(\mathbf{C}) \le M - 1$ after centering and the metric it induces embeds in at most $\min(M - 1,\, d)$ dimensions. The cell below MEASURES that floor against $\alpha R_\text{eff}$ instead of assuming it.

$R_\text{eff}$ is finite only within a connected component, which is what the anchor bond of the figure buys: $\mathcal{E}_\text{Tx}$ is a cycle and $\mathcal{E}_\text{Sc}$ is connected, so the single bond $t_0 \to \text{anchor} \to v^\text{Sc}_1$ joins the two layers whatever $\mathcal{E}_\text{Cross}$ happens to cover, and no pair is left at infinite resistance. $\mathcal{V}_\text{Tx}$ is the context window the LLM is served and no more — the $1024$ tokens FROM the last scene-graph block on, since the system preamble and any ICL graphs are text no crosslink reaches; $t_0$ is the token that opens that block. The crosslinks are laid down BIDIRECTIONALLY here, as the figure has them, since this stage has no prefill to keep causal — and $R_\text{eff}$ could not tell the difference in any case: $\bar{\mathbf{H}}^{(r)}$ symmetrizes $\mathbf{A}$, and transposing $\mathcal{G}$ merely conjugates $\big(\bar{\mathbf{L}}^{(r)}\big)^\dagger$, leaving its real part alone.

The charge is PINNED for this stage rather than learned. $R_\text{eff}$ is a function of $r$, so a learnable charge would let the model move its own target instead of meeting it; and $r > 0$ is what keeps the target well-scaled, since at $r = 0$ the magnetic Laplacian degenerates to the symmetric normalized Laplacian, whose kernel $\mathbf{D}_s^{1/2}\mathbf{1}$ must be projected out and whose near-null cycle mode inflates $R_\text{eff}$ by two orders of magnitude on the same graph.

In [ ]:
# Define the effective-resistance target and the covariance metric that must match it.
ALPHA = 1.0


@torch.no_grad()
def magnetic_resistance(graph, conv) -> Tensor:
    """R_eff(u, v) = (e_u - e_v)^H (L̄^(r))† (e_u - e_v) over the whole composite graph.

    The shift operator is `conv`'s OWN: `MagChebConv.__norm__` returns -H̄^(r) under
    shift='laplacian', so L̄^(r) is the identity plus what it hands back, and the target
    is the resistance of the very operator the network filters over. Hermitian L̄^(r)
    makes the form real — Im(L̄†_uv) cancels against L̄†_vu — and L̄^(r) ⪰ 0 makes it a
    squared distance; the clamp only absorbs the rounding of that cancellation.
    """
    assert conv.r_logit is None, 'a learnable charge would move the target under the model'
    n = graph.num_nodes
    edge_index, shift = conv.__norm__(graph.edge_index, n, graph.edge_weight,
                                      conv.normalization, dtype=torch.float)
    lap = torch.eye(n, dtype=torch.complex128, device=graph.edge_index.device)
    lap[edge_index[0], edge_index[1]] += shift.to(torch.complex128)
    pinv = torch.linalg.pinv(lap, hermitian=True)
    diag = pinv.diagonal().real
    return (diag[:, None] + diag[None, :] - 2 * pinv.real).clamp_min(0).float()


def gram_distances(gram: Tensor) -> Tensor:
    """G_nn + G_mm - 2G_nm for every pair — the squared distance a Gram matrix induces.
    Applied to C itself rather than to any factor of it, since C is the matrix the
    composite architecture injects; `torch.cdist` would route the same quantity through
    a square root whose gradient is undefined on the zero diagonal."""
    diag = gram.diagonal()
    return (diag[:, None] + diag[None, :] - 2 * gram).clamp_min(0)


def probe_covariance(model, graph) -> Tensor:
    """C = E_q[ΦΦᵀ] - ΨΨᵀ over EVERY node of `graph`, for Φ = Φ(q; L̄^(r), H).

    `covariance_token_block` already forms exactly this from the R-PEARL probe stack,
    and asking it for the whole graph rather than the leading c rows returns C in full,
    with C_tok as its token block. It centers before contracting — manifestly PSD, and
    free of the fp32 cancellation E[ΦΦᵀ] - ΨΨᵀ suffers once the mean dominates the
    fluctuation. Gradient reaches the MagNet backbone; the blocks are downstream of C.
    """
    C, _ = model.pe_model.covariance_token_block(graph, graph.num_nodes)
    return C


# C is the encoding this stage shapes, so the GT keeps its d_model width and its pure
# probe path; the charge is pinned so that the target is a fixed function of the topology.
model_type = 'gt'
gnn = create_gnn(model_type, learn_r=False).to(device)
conv = gnn.pe_model.pe_gcn.convs[0]
assert model_hparams['directed'], 'the resistance target is defined by the magnetic shift'
assert model_hparams['pe_pool'] == 'pe', 'C is read off the probe stack before the blocks'

# The composite graph of the figure: bidirectional crosslinks and the anchor bond.
res_composite = build_composite_graph(
    graph_file_by_name[graph_file], tokenizer, device=device,
    crosslink_bidirectional=True, anchor=True,
)
R = magnetic_resistance(res_composite, conv).double()
print(f"R_eff: {tuple(R.shape)} | Charge: {conv.r:.4f} | Asymmetry: {(R - R.T).abs().max():.2e} "
      f"| Diagonal: {R.diagonal().abs().max():.2e} | Min: {R.min():.4f}, "
      f"Mean: {R.mean():.4f}, Max: {R.max():.4f}")

# The floor: rank(C) ≤ M - 1 after centering, so the metric C induces embeds in at most
# min(M - 1, d_model) dimensions. Double-centering αR_eff recovers the Gram matrix of an
# exact embedding, whose leading eigenpairs are the best a C of that rank can do.
rank = min(model_hparams['num_samples'] - 1, model_hparams['d_model'])
target = ALPHA * R
gram = -0.5 * (target - target.mean(0, keepdim=True) - target.mean(1, keepdim=True)
               + target.mean())
lam, U = torch.linalg.eigh(gram)
floor = gram_distances((U[:, -rank:] * lam[-rank:].clamp_min(0)) @ U[:, -rank:].T)
print(f"Rank floor at rank {rank} of {R.shape[0]}: "
      f"{(floor - target).norm() / target.norm():.2e} relative error")

# The untrained C, to state the scale the loss starts from and the memory it costs.
with torch.no_grad():
    C = probe_covariance(gnn.eval(), res_composite).double()
print(f"C: {tuple(C.shape)} | Trace: {C.trace():.4f} | D(C) mean: "
      f"{gram_distances(C).mean():.4f} against αR_eff mean: {target.mean():.4f} "
      f"| Peak: {torch.cuda.max_memory_allocated() / 2 ** 30:.2f} GiB")
display(render_matrix(R[:5, :5]))

In [ ]:
# Preprocess the data. The split is §2's, so no scene graph leaks across it, and R is a
# function of the topology alone, so each graph carries its own target from here on.
def seed_resistance(graph, conv):
    """Attaches the effective-resistance target of `graph` under `conv`'s shift operator.
    R is a WITHIN-component quantity, so the anchor's claim — that the composite graph is
    one component — is asserted here rather than assumed."""
    plain = Data(edge_index=graph.edge_index, num_nodes=graph.num_nodes)
    assert nx.is_connected(to_networkx(plain, to_undirected=True)), \
        'the composite graph is disconnected, so its resistances are not comparable'
    graph.R = magnetic_resistance(graph, conv)

    return graph


res_kwargs = dict(crosslink_bidirectional=True, anchor=True)
train_res = [seed_resistance(g, conv) for g in generate_data(train_keys, **res_kwargs)]
val_res = [seed_resistance(g, conv) for g in generate_data(val_keys, **res_kwargs)]
test_res = [seed_resistance(g, conv) for g in generate_data(test_keys, **res_kwargs)]

# One N×N fp32 target per graph is this stage's whole standing allocation; state it.
graphs_res = train_res + val_res + test_res
print(f"Resistance Targets: {len(train_res)}/{len(val_res)}/{len(test_res)} graphs "
      f"| Nodes: {min(g.num_nodes for g in graphs_res)}-{max(g.num_nodes for g in graphs_res)} "
      f"| Held: {sum(g.R.numel() for g in graphs_res) * 4 / 2 ** 30:.2f} GiB")

In [ ]:
# Train the Graph Transformer so that C reproduces the composite graph's resistances.
batch_size = 4
val_freq = 5
epochs = 50
es_patience = 5
train_resistance = True


def test_loop_resistance(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss, error_norm, token_norm = 0, 0, 0

    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            preds = gram_distances(probe_covariance(model, graph))
            target, c = ALPHA * graph.R, graph.num_token_nodes
            test_loss += loss_fn(preds, target).item()
            error_norm += ((preds - target).norm() / target.norm()).item()
            # C_tok is the block the LLM is served, so it is scored on its own too.
            token_norm += ((preds[:c, :c] - target[:c, :c]).norm()
                           / target[:c, :c].norm()).item()

    test_loss /= size
    error_norm /= size
    token_norm /= size
    print(f"Test Error: \n Rel Err: {error_norm:.3f}, Token Err: {token_norm:.3f} "
          f"| Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/error_norm': error_norm,
            f'{wandb_prefix}/token_error_norm': token_norm,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, error_norm


def train_loop_resistance(train_dataloader, val_dataloader, test_dataloader, model,
              loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    best_val, best_state, bad_runs = float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    # The pinned charge and α belong there too, since they are what fix the target.
    run = init_wandb('resistance_regression', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        'plans_per_graph': plans_per_graph,
        'charge': float(conv.r), 'alpha': ALPHA,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq + 1}\n=============")
            val_loss, _ = test_loop_resistance(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if val_loss < best_val - 1e-3:
                best_val, bad_runs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model.train()

        print(f"=============\nEpoch #{i + 1}\n=============")
        optimizer.zero_grad()
        pending = 0
        order = torch.randperm(size)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss. The second moment IS the prediction — no
            # readout head stands between the probe response and the metric it carries.
            graph = train_dataloader.dataset[idx]
            preds = gram_distances(probe_covariance(model, graph))
            loss = loss_fn(preds, ALPHA * graph.R)

            # Backpropagation.
            (loss / batch_size).backward()
            pending += 1

            # Optimization and results.
            if pending == batch_size:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                pending = 0

                global_step += 1
                loss, current = loss.item(), j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

        if pending:
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            pending = 0

    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test the finished model.
    test_loop_resistance(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# MSELoss is the stated sum divided by N², so graphs of different size weigh equally.
loss_fn = nn.MSELoss()
train_dataloader = DataLoader(train_res, batch_size=batch_size)
val_dataloader = DataLoader(val_res, batch_size=batch_size)
test_dataloader = DataLoader(test_res, batch_size=batch_size)
if train_resistance:
    # C is read off the probe stack BEFORE the blocks, so the loss reaches the MagNet
    # backbone and nothing else; the optimizer states that scope rather than implying it.
    optimizer = torch.optim.AdamW(gnn.pe_model.parameters(), lr=3e-4, betas=(0.9, 0.95),
                                  weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_resistance(train_dataloader, val_dataloader, test_dataloader, gnn,
                          loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler
    gc.collect()